# HuggingFace Genomics Hands-On Lab
## DNA Sequence Analysis with Foundation Models

**Author:** Ikram Ullah, KAUST Bioinformatics Platform

---

## Welcome!

This hands-on lab will guide you through the practical aspects of working with **Genomic Foundation Models (GFMs)** using the Hugging Face ecosystem. By the end of this session, you'll be comfortable loading pre-trained DNA models, extracting embeddings, and fine-tuning classifiers for genomic tasks.

---

## Learning Objectives

By completing this lab, you will be able to:

1. **Encode DNA sequences** as PyTorch tensors using different representations (integer, one-hot)
2. **Load pre-trained genomic models** (DNABERT-2, Nucleotide Transformer) from Hugging Face Hub
3. **Extract embeddings** from DNA sequences using foundation models
4. **Use the Hugging Face Inference API** for quick prototyping without local GPU
5. **Fine-tune a classifier** on a real genomic task (promoter detection)
6. **Evaluate model performance** using standard metrics

---

## Prerequisites

- Basic Python and PyTorch knowledge (covered in the PyTorch notebook)
- Understanding of DNA sequences (A, T, C, G nucleotides)
- GPU access recommended (but CPU will work for small examples)

---

## Lab Structure

| Part | Topic | Time |
|------|-------|------|
| 1 | PyTorch Tensors for DNA | 10 min |
| 2 | Loading Genomic Models | 15 min |
| 3 | Exploring Hugging Face Hub | 10 min |
| 4 | Inference API | 15 min |
| 5 | Fine-tuning on Genomic Data | 30 min |

---

## Key Concept: The GFM Pipeline

```
DNA Sequence → Tokenization → Embedding → Transformer → Task Head → Prediction
    "ATCG"      [3,4,5,6]     [0.1,...]    [layers]    [linear]    [0.9, 0.1]
```

Let's begin!

---

## Setup: Install and Import Required Packages

Before we start, let's ensure all necessary packages are installed and imported. This cell sets up:

- **PyTorch**: The deep learning framework (with CUDA support if GPU available)
- **Transformers**: Hugging Face library for loading pre-trained models
- **Visualization libraries**: matplotlib, umap-learn for plotting embeddings

> **Note**: If you're using the workshop environment, these are already installed. Otherwise, uncomment the pip install lines.

In [1]:
# ============================================================
# INSTALLATION (uncomment if packages not already installed)
# ============================================================
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip install matplotlib scikit-learn ipython 
# !pip install transformers umap-learn evaluate 
# !pip uninstall -y triton  # Remove if causing conflicts

In [3]:
# ============================================================
# IMPORTS - Core libraries for this lab
# ============================================================
import torch                           # Deep learning framework
import numpy as np                     # Numerical operations
import matplotlib.pyplot as plt        # Visualization
from transformers import AutoTokenizer, AutoModel, AutoConfig  # Hugging Face
from sklearn.decomposition import PCA  # Dimensionality reduction
import umap                            # Non-linear embedding visualization
import time                            # For timing comparisons
import os                              # Environment variables (HF_TOKEN)

# Verify PyTorch installation and GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.9.1+cu128
CUDA available: True
GPU device: Tesla V100-SXM2-32GB


### Device Selection

Deep learning computations run much faster on GPUs. This cell automatically selects the best available device:
- **CUDA**: NVIDIA GPU (preferred for speed)
- **CPU**: Fallback if no GPU available

> **Important**: Always ensure your model and data are on the **same device** to avoid errors!

In [4]:
# Select the best available compute device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


---

## Part 1: Encoding DNA as PyTorch Tensors

Before we can feed DNA sequences to neural networks, we need to convert them from strings to numerical tensors. This section covers three encoding approaches used in genomic models:

1. **Integer encoding**: Map each nucleotide to a number (A=0, T=1, C=2, G=3)
2. **One-hot encoding**: Represent each nucleotide as a binary vector
3. **Byte Pair Encoding (BPE)**: Subword tokenization used by models like DNABERT-2

It also discusses:
- **Device management**: Moving tensors between CPU and GPU

> **Why this matters**: Understanding these encoding methods helps you choose the right model for your task and debug tokenization issues.

---

### 1.1 Integer Encoding

The simplest encoding: each nucleotide becomes an integer index.

In [5]:
# Example DNA sequence
dna = "ATCGATCGATCG"

# Create a mapping: nucleotide -> integer
mapping = {'A': 0, 'T': 1, 'C': 2, 'G': 3}

# Convert each nucleotide to its integer representation
encoded = [mapping[base] for base in dna]

# Convert Python list to PyTorch tensor
tensor = torch.tensor(encoded)

# Display results
print(f"Original DNA:  {dna}")
print(f"Integer codes: {encoded}")
print(f"PyTorch tensor: {tensor}")
print(f"Tensor shape:   {tensor.shape}  # (sequence_length,)")

Original DNA:  ATCGATCGATCG
Integer codes: [0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3]
PyTorch tensor: tensor([0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3])
Tensor shape:   torch.Size([12])  # (sequence_length,)


### 1.2 One-Hot Encoding

One-hot encoding represents each nucleotide as a 4-dimensional binary vector:
- A = [1, 0, 0, 0]
- T = [0, 1, 0, 0]
- C = [0, 0, 1, 0]
- G = [0, 0, 0, 1]

This representation is useful because:
- No implicit ordering between nucleotides (unlike integer encoding where G=3 > A=0)
- Works well with traditional ML models
- Easy to interpret

In [6]:
def one_hot_encode(seq):
    """
    Convert a DNA sequence to one-hot encoding.
    
    Args:
        seq: DNA string (e.g., "ATCG")
    
    Returns:
        Tensor of shape (seq_length, 4)
    """
    mapping = {'A': 0, 'T': 1, 'C': 2, 'G': 3}
    one_hot = torch.zeros(len(seq), 4)  # Initialize with zeros
    for i, base in enumerate(seq):
        one_hot[i, mapping[base]] = 1   # Set the appropriate position to 1
    return one_hot

# Encode a single sequence
encoded = one_hot_encode("ATCG")
print("One-hot encoding of 'ATCG':")
print(encoded)
print(f"Shape: {encoded.shape}  # (seq_length=4, nucleotides=4)")

# Batch multiple sequences using torch.stack()
print("\n--- Batching multiple sequences ---")
sequences = ["ATCG", "GCTA"]
batch = torch.stack([one_hot_encode(s) for s in sequences])
print(f"Batch shape: {batch.shape}  # (batch_size=2, seq_length=4, nucleotides=4)")

One-hot encoding of 'ATCG':
tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]])
Shape: torch.Size([4, 4])  # (seq_length=4, nucleotides=4)

--- Batching multiple sequences ---
Batch shape: torch.Size([2, 4, 4])  # (batch_size=2, seq_length=4, nucleotides=4)


### 1.3 Byte Pair Encoding (BPE)

**Byte Pair Encoding (BPE)** is a subword tokenization algorithm originally developed for data compression, now widely used in NLP and genomic models like **DNABERT-2**.

#### How BPE Works

BPE iteratively merges the most frequent pairs of characters (or tokens) in the training data:

1. Start with individual characters as the initial vocabulary (e.g., `A`, `T`, `C`, `G`)
2. Count all adjacent pairs and find the most frequent one
3. Merge that pair into a new token and add it to the vocabulary
4. Repeat until the desired vocabulary size is reached

**Example**: If `"AT"` appears frequently, BPE creates a new token `"AT"`. Later, if `"ATG"` is common, it becomes another token.

#### Why BPE for Genomics?

| Advantage | Description |
|-----------|-------------|
| **Variable-length tokens** | Unlike fixed k-mers (e.g., 6-mers), BPE learns optimal token lengths from data |
| **Smaller vocabulary** | More efficient than enumerating all possible k-mers (4^k grows exponentially) |
| **Handles rare patterns** | Can fall back to smaller subwords for unseen sequences |
| **Cross-species generalization** | Learns patterns that may transfer across organisms |

#### BPE vs K-mer Tokenization

| Aspect | BPE (DNABERT-2) | K-mer (Nucleotide Transformer) |
|--------|-----------------|--------------------------------|
| Token length | Variable (learned from data) | Fixed (e.g., 6 nucleotides) |
| Vocabulary size | Configurable (e.g., 4,096) | Fixed at 4^k (4,096 for 6-mers) |
| Flexibility | High - adapts to patterns | Lower - all k-mers enumerated |
| Rare sequences | Falls back to smaller tokens | May use `<unk>` token |



#### BPE for DNA: Minimal Example

**Training sequences**
```text
ATGATG
ATGATG
GCGC
```

**Step 0 – initial tokens**
- Vocab: `{A, T, G, C}`
- ATGATG → `A T G A T G`  
- GCGC → `G C G C`

**Step 1 – merge most frequent pair `AT`**
- New vocab: `{A, T, G, C, AT}`
- ATGATG → `AT G AT G`  
- GCGC → `G C G C`  (unchanged)

**Step 2 – merge most frequent pair `AT G`**
- New vocab: `{A, T, G, C, AT, ATG}`
- ATGATG → `ATG ATG`  
- GCGC → `G C G C`  (still unchanged)

You can stop here and show the trade‑off:

```python
# With vocab {A,T,G,C,AT}
"ATGATG" -> ["AT", "G", "AT", "G"]   # 4 tokens
"GCGC"   -> ["G", "C", "G", "C"]     # 4 tokens

# With vocab {A,T,G,C,AT,ATG}
"ATGATG" -> ["ATG", "ATG"]           # 2 tokens
"GCGC"   -> ["G", "C", "G", "C"]     # 4 tokens  (no merge learned here)
```

This shows:
- BPE is frequency‑driven (repeats in ATGATG get merged, GCGC doesn’t yet).  
- Larger vocab can give fewer tokens for frequent patterns, while rare patterns stay character-level.

In [7]:
# ============================================================
# BPE Tokenization Example with DNABERT-2
# ============================================================
from transformers import AutoTokenizer

# Load DNABERT-2 tokenizer (uses BPE)
bpe_tokenizer = AutoTokenizer.from_pretrained(
    "zhihan1996/DNABERT-2-117M", 
    trust_remote_code=True
)

# Example DNA sequence
dna_sequence = "ATCGATCGATCGATCG"

# Tokenize the sequence
tokens = bpe_tokenizer.tokenize(dna_sequence)
token_ids = bpe_tokenizer.encode(dna_sequence)

print("BPE Tokenization Example")
print("=" * 50)
print(f"Input sequence:  {dna_sequence}")
print(f"Sequence length: {len(dna_sequence)} nucleotides")
print(f"\nTokens:          {tokens}")
print(f"Number of tokens: {len(tokens)}")
print(f"\nToken IDs:       {token_ids}")

# Decode back to verify
decoded = bpe_tokenizer.decode(token_ids, skip_special_tokens=True)
print(f"\nDecoded back:    {decoded}")

# Show vocabulary size
print(f"\n--- Tokenizer Info ---")
print(f"Vocabulary size: {bpe_tokenizer.vocab_size:,}")

BPE Tokenization Example
Input sequence:  ATCGATCGATCGATCG
Sequence length: 16 nucleotides

Tokens:          ['A', 'TCGA', 'TCGA', 'TCGA', 'TC', 'G']
Number of tokens: 6

Token IDs:       [1, 5, 359, 359, 359, 16, 7, 2]

Decoded back:    A TCGA TCGA TCGA TC G

--- Tokenizer Info ---
Vocabulary size: 4,096


In [8]:
# ============================================================
# Compare BPE vs K-mer Tokenization
# ============================================================

# Load a k-mer tokenizer (Nucleotide Transformer uses 6-mers)
kmer_tokenizer = AutoTokenizer.from_pretrained(
    "InstaDeepAI/nucleotide-transformer-v2-50m-3mer-multi-species",
    trust_remote_code=True
)

# Same DNA sequence
dna_sequence = "ATCGATCGATCGATCG"

# Tokenize with both
bpe_tokens = bpe_tokenizer.tokenize(dna_sequence)
kmer_tokens = kmer_tokenizer.tokenize(dna_sequence)

print("Comparison: BPE vs K-mer Tokenization")
print("=" * 50)
print(f"Input: {dna_sequence} ({len(dna_sequence)} bp)\n")

print("BPE (DNABERT-2):")
print(f"  Tokens: {bpe_tokens}")
print(f"  Count:  {len(bpe_tokens)} tokens")

print("\nK-mer (Nucleotide Transformer):")
print(f"  Tokens: {kmer_tokens}")
print(f"  Count:  {len(kmer_tokens)} tokens")

print("\n--- Key Difference ---")
print(f"BPE learned variable-length tokens from data patterns.")
print(f"K-mer uses fixed-length tokens (each token = 3 nucleotides here).")

Comparison: BPE vs K-mer Tokenization
Input: ATCGATCGATCGATCG (16 bp)

BPE (DNABERT-2):
  Tokens: ['A', 'TCGA', 'TCGA', 'TCGA', 'TC', 'G']
  Count:  6 tokens

K-mer (Nucleotide Transformer):
  Tokens: ['ATC', 'GAT', 'CGA', 'TCG', 'ATC', 'G']
  Count:  6 tokens

--- Key Difference ---
BPE learned variable-length tokens from data patterns.
K-mer uses fixed-length tokens (each token = 3 nucleotides here).


### 1.4 Moving Tensors Between Devices

PyTorch tensors can live on either CPU or GPU memory. For fast computation:
- Models and input data must be on the **same device**
- Use `.to(device)` to move tensors
- Use `.cpu()` to bring results back to CPU (needed for NumPy, plotting, etc.)

> **Common Error**: `RuntimeError: Expected all tensors to be on the same device` — this means your model and data are on different devices!

In [9]:
# Select device (already done above, but shown here for clarity)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create a tensor on CPU (default)
t_cpu = torch.randn(100, 512)  # Random tensor, shape (100, 512)
print(f"Created on: {t_cpu.device}")

# Move to GPU (if available)
t_gpu = t_cpu.to(device)
print(f"Moved to:   {t_gpu.device}")

# Bring back to CPU (required for NumPy/matplotlib operations)
t_back = t_gpu.cpu()
print(f"Back to:    {t_back.device}")

# Convert to NumPy (only works on CPU tensors)
numpy_array = t_back.numpy()
print(f"NumPy shape: {numpy_array.shape}")

Created on: cpu
Moved to:   cuda:0
Back to:    cpu
NumPy shape: (100, 512)


---

## Recap: What We Learned This Morning

In the morning session, you worked with **HyenaDNA** and **Nucleotide Transformer** to:

1. **Extract embeddings** from real GENCODE transcripts (protein-coding vs lncRNA)
2. **Apply mean pooling** to get a single vector per sequence
3. **Visualize** with UMAP and t-SNE — seeing how the model separates biotypes in embedding space

**Key takeaway**: These models learn a "DNA grammar" that captures biological signals, even without explicit labels.

---

## This Afternoon: The Hugging Face Ecosystem

Now we'll explore the **tools and workflows** that make working with these models practical:

| Morning (Done ✓) | Afternoon (Now) |
|------------------|-----------------|
| What embeddings are | How to access 1000s of models via HF Hub |
| Manual extraction | Streamlined APIs and pipelines |
| Single model deep-dive | Comparing models (DNABERT-2 vs NT) |
| Visualization | Fine-tuning for real tasks |

Let's start by loading some models!

---

## Part 2: Loading Pre-trained Genomic Models

Now we'll load real genomic foundation models from the Hugging Face Hub. The key classes are:

- **AutoTokenizer**: Handles DNA sequence → token IDs conversion
- **AutoModel**: Loads the pre-trained transformer encoder
- **AutoConfig**: Access model configuration (hidden size, layers, etc.)

We'll compare two popular models:
1. **DNABERT-2**: BERT-based model with BPE tokenization
2. **Nucleotide Transformer**: ESM-based model with k-mer tokenization

> **Note**: The first time you run these cells, models will be downloaded (~500MB-2GB). This only happens once.

---

### 2.1 Load DNABERT-2 (117M parameters)

DNABERT-2 uses **Byte-Pair Encoding (BPE)** tokenization and **ALiBi** position embeddings for handling variable-length sequences.

<details>
<summary>Byte-Pair Encoding (BPE)</summary>

Byte-Pair Encoding is a subword tokenization method that iteratively merges the most frequent pairs of symbols in a sequence. In DNABERT-2, BPE allows variable-length DNA sequences to be tokenized efficiently without relying on fixed k-mers, improving flexibility, vocabulary efficiency, and generalization to unseen sequences.

</details>

<details>
<summary>ALiBi (Attention with Linear Biases)</summary>

ALiBi is a position encoding technique that adds a linear bias to attention scores based on the relative distance between tokens. Instead of using explicit positional embeddings, ALiBi enables transformers to naturally handle variable-length sequences and extrapolate to longer inputs than seen during training, making it well-suited for genomic sequences of varying lengths.

</details>

In [10]:
from transformers import AutoTokenizer, AutoModel, BertConfig
from termcolor import colored
from IPython.display import display, Markdown

# Model identifier on Hugging Face Hub
model_name = "zhihan1996/DNABERT-2-117M"

# Load the three key components:
# 1. Tokenizer: converts DNA strings to token IDs
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# 2. Config: model architecture details
config = BertConfig.from_pretrained(model_name)

# 3. Model: the actual neural network weights
model = AutoModel.from_pretrained(model_name, trust_remote_code=True, config=config)

# Count total parameters
params = sum(p.numel() for p in model.parameters())

# Display model summary in a nice table
info_md = f"""
### DNABERT-2 Model Summary

| Property | Value |
|:---------|:------|
| **Model ID** | `{model_name}` |
| **Architecture** | BERT + ALiBi (Attention with Linear Biases) |
| **Tokenizer Type** | `{type(tokenizer).__name__}` (BPE-based) |
| **Vocab Size** | {tokenizer.vocab_size:,} |
| **Max Sequence Length** | {config.max_position_embeddings} |
| **Hidden Size** | {config.hidden_size} |
| **Number of Layers** | {config.num_hidden_layers} |
| **Attention Heads** | {config.num_attention_heads} |
| **Total Parameters** | **{params:,}** (~{params/1e6:.0f}M) |

> **Tip**: The "not initialized" warning is normal - we're loading an encoder, not a task-specific model.
"""
display(Markdown(info_md))

Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



### DNABERT-2 Model Summary

| Property | Value |
|:---------|:------|
| **Model ID** | `zhihan1996/DNABERT-2-117M` |
| **Architecture** | BERT + ALiBi (Attention with Linear Biases) |
| **Tokenizer Type** | `PreTrainedTokenizerFast` (BPE-based) |
| **Vocab Size** | 4,096 |
| **Max Sequence Length** | 512 |
| **Hidden Size** | 768 |
| **Number of Layers** | 12 |
| **Attention Heads** | 12 |
| **Total Parameters** | **117,068,544** (~117M) |

> **Tip**: The "not initialized" warning is normal - we're loading an encoder, not a task-specific model.


### 2.2 Load Nucleotide Transformer (500M parameters)

The Nucleotide Transformer (NT) uses **k-mer tokenization** (typically 6-mers) and was trained on the human reference genome. It's larger than DNABERT-2 but often achieves better performance on genomic tasks.

In [11]:
# Nucleotide Transformer model ID
nt_name = "InstaDeepAI/nucleotide-transformer-500m-human-ref"

# Load tokenizer, config, and model
nt_tokenizer = AutoTokenizer.from_pretrained(nt_name, trust_remote_code=True)
nt_config = AutoConfig.from_pretrained(nt_name)

# Load model, move to GPU, and set to evaluation mode
# .eval() disables dropout for consistent inference
nt_model = AutoModel.from_pretrained(nt_name, trust_remote_code=True).to(device).eval()

# Count parameters
nt_params = sum(p.numel() for p in nt_model.parameters())

# Display model summary
nt_md = f"""
### Nucleotide Transformer Summary

| Property | Value |
|:---------|:------|
| **Model ID** | `{nt_name}` |
| **Architecture** | Transformer (ESM-based, RoFormer variant) |
| **Pretraining** | Masked Language Modeling on human reference genome |
| **Tokenizer Type** | `{type(nt_tokenizer).__name__}` (k-mer based) |
| **Vocab Size** | {nt_tokenizer.vocab_size:,} |
| **Max Sequence Length** | {nt_config.max_position_embeddings:,} |
| **Hidden Size** | {nt_config.hidden_size} |
| **Number of Layers** | {nt_config.num_hidden_layers} |
| **Attention Heads** | {nt_config.num_attention_heads} |
| **Total Parameters** | **{nt_params:,}** (~{nt_params/1e6:.0f}M) |

> **Comparison**: NT has ~4x more parameters than DNABERT-2 and larger hidden dimensions.
"""
display(Markdown(nt_md))

Some weights of EsmModel were not initialized from the model checkpoint at InstaDeepAI/nucleotide-transformer-500m-human-ref and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



### Nucleotide Transformer Summary

| Property | Value |
|:---------|:------|
| **Model ID** | `InstaDeepAI/nucleotide-transformer-500m-human-ref` |
| **Architecture** | Transformer (ESM-based, RoFormer variant) |
| **Pretraining** | Masked Language Modeling on human reference genome |
| **Tokenizer Type** | `EsmTokenizer` (k-mer based) |
| **Vocab Size** | 4,107 |
| **Max Sequence Length** | 1,002 |
| **Hidden Size** | 1280 |
| **Number of Layers** | 24 |
| **Attention Heads** | 20 |
| **Total Parameters** | **480,438,241** (~480M) |

> **Comparison**: NT has ~4x more parameters than DNABERT-2 and larger hidden dimensions.


---

## Part 3: Exploring the Hugging Face Model Hub

The [Hugging Face Model Hub](https://huggingface.co/models) is a repository of thousands of pre-trained models. For genomics, you can find:

- **DNABERT variants**: BPE-tokenized DNA models
- **Nucleotide Transformer variants**: Different sizes (50M to 2.5B parameters)
- **Species-specific models**: Human, multi-species, etc.

### Key Information to Look For

When selecting a model, check the **Model Card** for:

| What to Check | Why It Matters |
|---------------|----------------|
| **Tokenization method** | Affects how sequences are encoded (BPE vs k-mer) |
| **Max sequence length** | Determines if your sequences will fit |
| **Model size** | Larger = more powerful but slower |
| **Training data** | Human-only vs multi-species |

### Exercise: Browse the Model Hub

**Instructions**: Open https://huggingface.co/models in a new tab and try these searches:

1. Search for `"DNABERT"` → Find `zhihan1996/DNABERT-2-117M`
2. Search for `"nucleotide transformer"` → Find InstaDeep's model variants
3. Search for `"genomic"` → Discover other DNA/RNA models

**Questions to answer**:
- How many downloads does DNABERT-2 have?
- What is the vocabulary size of the Nucleotide Transformer?
- Can you find a model for RNA sequences?

### Exercise: Analyze a Model Card

Visit the [HyenaDNA Model Card](https://huggingface.co/LongSafari/hyenadna-medium-160k-seqlen-hf/tree/main) and find:

| Question | Answer |
|----------|--------|
| Vocabulary Size? | 12 |
| Max sequence length? | 1.6 Million tokens |
| Architecture? | HyenaDNAForCausalLM |
| Parameters? | 14.2M |

---

## Part 4: Using the Hugging Face Inference API

The **Inference API** lets you run models in the cloud without local GPU setup. This is useful for:

- Quick prototyping and testing
- When you don't have local GPU access
- Comparing different models easily

### How It Works

```
Your Code  →  HTTP Request  →  HF Servers  →  Model Inference  →  Response
```

> **Requirement**: You need a free Hugging Face account and API token. Set your token as the `HF_TOKEN` environment variable.

---

### 4.1 Authentication Setup

Make sure your `HF_TOKEN` is set in your environment. The API uses this for authentication.

### 4.2 Masked Language Modeling (Fill-in-the-Blank)

We'll use the API to perform **Masked Language Modeling (MLM)** using Nucleotide Transformer model. The model predicts what nucleotides should fill a `<mask>` token in a sequence. This demonstrates how the model has learned DNA grammar.

In [12]:
import requests
import pandas as pd
from IPython.display import display, HTML

def query_api(seq, model_id, token):
    """
    Query the Hugging Face Inference API.
    
    Args:
        seq: Input sequence (may contain <mask> for MLM)
        model_id: Hugging Face model identifier
        token: HF API token
    
    Returns:
        JSON response with predictions, or None if error
    """
    url = f"https://router.huggingface.co/hf-inference/models/{model_id}"
    headers = {"Authorization": f"Bearer {token}"}
    
    response = requests.post(url, headers=headers, json={"inputs": seq})
    
    if response.status_code != 200:
        print(f"API Error {response.status_code}: {response.text}")
        return None
    
    try:
        return response.json()
    except Exception as e:
        print("Response not in JSON format:", e)
        print(response.text[:300])
        return None

**How to find token for masking**

Check the vocabulary file on [`InstaDeepAI/nucleotide-transformer-500m-human-ref` model page](https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-500m-multi-species). We see it is `<mask>`

In [13]:
# Example: Ask the model to fill in the <mask> token
masked_seq = "ATCGATCG<mask>ATCG"
model_id = "InstaDeepAI/nucleotide-transformer-500m-human-ref"

# Query the API
result = query_api(masked_seq, model_id, os.getenv('HF_TOKEN'))

# Display results in a nice table
if result:
    df = pd.DataFrame(result)[["token_str", "score", "sequence"]]
    df.rename(columns={
        "token_str": "Predicted DNA (replacing <mask>)",
        "score": "Confidence",
        "sequence": "Full Sequence"
    }, inplace=True)

    print(f"Input sequence: {masked_seq}")
    print(f"Model: {model_id}")
    print(f"\nTop 5 predictions for the masked region:")
    
    display(HTML(df.style
        .bar(subset=["Confidence"], color='#a6cee3')
        .format({"Confidence": "{:.1%}"})
        .set_caption("Model predictions ranked by confidence")
        .to_html()
    ))
else:
    print("API call failed - check your HF_TOKEN")

Input sequence: ATCGATCG<mask>ATCG
Model: InstaDeepAI/nucleotide-transformer-500m-human-ref

Top 5 predictions for the masked region:


,Predicted DNA (replacing ),Confidence,Full Sequence
0,AGCCTT,3.3%,ATCGAT C G AGCCTT A T C G
1,TTTCCT,2.4%,ATCGAT C G TTTCCT A T C G
2,CAGCTT,2.4%,ATCGAT C G CAGCTT A T C G
3,CGTCTT,2.3%,ATCGAT C G CGTCTT A T C G
4,CTTTTC,2.2%,ATCGAT C G CTTTTC A T C G


**Understanding the Results**:

Each row shows a **k-mer** (6 nucleotides) that the model predicts should replace the `<mask>` token. The model has learned patterns from billions of nucleotides and uses context to make predictions - similar to how language models predict the next word in a sentence.

> **Key Insight**: The model doesn't just predict random nucleotides - it considers the surrounding context to make biologically plausible predictions.

### 4.3 Local vs API Inference: A Comparison

Let's compare running inference **locally** (on your GPU) vs **via the API** (on Hugging Face servers). This will help you understand the trade-offs:

| Aspect | Local Inference | API Inference |
|--------|-----------------|---------------|
| **Speed** | Fast (once loaded) | Network latency |
| **Setup** | Requires GPU + installation | Just an API token |
| **Cost** | Your hardware | Free tier limits |
| **Flexibility** | Full control | Limited to supported models |

In [14]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM
import requests
import pandas as pd
from IPython.display import display, HTML
import logging
import time

# Suppress verbose logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = "InstaDeepAI/nucleotide-transformer-500m-human-ref"
print(f"Model: {model_id}")
print(f"Device: {device}\n")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# ============================================================
# PART A: Local Embedding Extraction
# ============================================================
print("=" * 50)
print("Part A: Extract Embeddings Locally")
print("=" * 50)

embed_model = AutoModel.from_pretrained(model_id, trust_remote_code=True).to(device).eval()

seq_plain = ["ATCGATCGATCG"]
tokens_plain = tokenizer(seq_plain, return_tensors="pt", padding=True, truncation=True).to(device)

with torch.no_grad():
    outputs = embed_model(**tokens_plain)
    embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()

print(f"Embedding shape: {embeddings.shape}")
print(f"This is a {embeddings.shape[1]}-dimensional representation of the sequence\n")

# ============================================================
# PART B: Local MLM Prediction
# ============================================================
print("=" * 50)
print("Part B: Local Masked Language Modeling")
print("=" * 50)

# Load MLM model
mlm_model = AutoModelForMaskedLM.from_pretrained(model_id, trust_remote_code=True).to(device).eval()

# Masked input
seq_masked = ["ATCGATCG<mask>ATCG"]
tokens_masked = tokenizer(seq_masked, return_tensors="pt", padding=True, truncation=True).to(device)

# Time the inference
t0 = time.time()
with torch.no_grad():
    outputs = mlm_model(**tokens_masked)
    logits = outputs.logits
local_time = time.time() - t0

# Find mask position and get predictions
mask_idx = (tokens_masked["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)
mask_logits = logits[mask_idx]

# Get top predictions
probs = F.softmax(mask_logits, dim=-1)
top_k = torch.topk(probs, 5, dim=-1)

print(f"Local inference time: {local_time:.3f}s\n")
print("Top 5 predictions for <mask>:")
for i, (tok_id, score) in enumerate(zip(top_k.indices[0].tolist(), top_k.values[0].tolist())):
    token = tokenizer.decode([tok_id]).strip()
    print(f"  {i+1}. '{token}' ({score*100:.1f}%)")

# ============================================================
# PART C: API Inference (for comparison)
# ============================================================
print("\n" + "=" * 50)
print("Part C: API Inference (Cloud)")
print("=" * 50)

t0 = time.time()
api_result = query_api(seq_masked[0], model_id, os.getenv('HF_TOKEN'))
api_time = time.time() - t0

if api_result:
    print(f"API inference time: {api_time:.3f}s\n")
    print("Top 5 predictions from API:")
    for i, r in enumerate(api_result[:5]):
        print(f"  {i+1}. '{r['token_str']}' ({r['score']*100:.1f}%)")
else:
    print("API call failed")

Model: InstaDeepAI/nucleotide-transformer-500m-human-ref
Device: cuda

Part A: Extract Embeddings Locally


The following layers were not sharded: contact_head.regression.bias, encoder.layer.*.output.dense.bias, contact_head.regression.weight, encoder.layer.*.intermediate.dense.weight, encoder.emb_layer_norm_after.weight, encoder.layer.*.attention.self.key.weight, embeddings.word_embeddings.weight, encoder.layer.*.LayerNorm.weight, pooler.dense.weight, encoder.layer.*.attention.LayerNorm.weight, encoder.layer.*.output.dense.weight, encoder.layer.*.attention.LayerNorm.bias, encoder.layer.*.LayerNorm.bias, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.attention.self.value.bias, encoder.layer.*.attention.output.dense.bias, pooler.dense.bias, encoder.layer.*.attention.self.query.weight, encoder.emb_layer_norm_after.bias, encoder.layer.*.attention.self.key.bias, encoder.layer.*.attention.self.query.bias, encoder.layer.*.attention.self.value.weight, encoder.layer.*.intermediate.dense.bias, embeddings.position_embeddings.weight


Embedding shape: (1, 1280)
This is a 1280-dimensional representation of the sequence

Part B: Local Masked Language Modeling


The following layers were not sharded: esm.encoder.layer.*.attention.self.value.bias, lm_head.dense.weight, lm_head.layer_norm.weight, esm.encoder.layer.*.attention.output.dense.weight, esm.encoder.layer.*.attention.output.dense.bias, esm.encoder.layer.*.LayerNorm.weight, esm.encoder.layer.*.output.dense.weight, esm.embeddings.word_embeddings.weight, esm.encoder.layer.*.intermediate.dense.weight, esm.encoder.emb_layer_norm_after.weight, esm.encoder.layer.*.attention.self.query.weight, lm_head.bias, lm_head.dense.bias, esm.encoder.layer.*.attention.self.query.bias, esm.contact_head.regression.bias, lm_head.decoder.weight, esm.encoder.layer.*.attention.LayerNorm.weight, esm.encoder.layer.*.attention.self.value.weight, esm.encoder.emb_layer_norm_after.bias, lm_head.layer_norm.bias, esm.encoder.layer.*.attention.LayerNorm.bias, esm.encoder.layer.*.intermediate.dense.bias, esm.contact_head.regression.weight, esm.encoder.layer.*.LayerNorm.bias, esm.embeddings.position_embeddings.weight, esm.

Local inference time: 0.016s

Top 5 predictions for <mask>:
  1. 'AGCCTT' (3.3%)
  2. 'TTTCCT' (2.4%)
  3. 'CAGCTT' (2.4%)
  4. 'CGTCTT' (2.3%)
  5. 'CTTTTC' (2.2%)

Part C: API Inference (Cloud)
API inference time: 0.794s

Top 5 predictions from API:
  1. 'AGCCTT' (3.3%)
  2. 'TTTCCT' (2.4%)
  3. 'CAGCTT' (2.4%)
  4. 'CGTCTT' (2.3%)
  5. 'CTTTTC' (2.2%)


In [16]:
import pandas as pd
from IPython.display import display, HTML

# Summary comparison table
print("=" * 50)
print("Summary: Local vs API Inference")
print("=" * 50)

comparison_data = {
    "Aspect": ["Runtime", "Setup", "Best For", "Limitations"],
    "Local GPU": [
        f"{local_time:.3f}s",
        "Requires GPU + packages",
        "Production, large batches",
        "Hardware cost"
    ],
    "HF API": [
        f"{api_time:.3f}s",
        "Just an API token",
        "Prototyping, small jobs",
        "Rate limits, network latency"
    ]
}

df = pd.DataFrame(comparison_data)
display(df)

print(f"Local is {api_time/local_time:.1f}x faster than API for this example.")

Summary: Local vs API Inference


,Aspect,Local GPU,HF API
0,Runtime,0.016s,0.794s
1,Setup,Requires GPU + packages,Just an API token
2,Best For,"Production, large batches","Prototyping, small jobs"
3,Limitations,Hardware cost,"Rate limits, network latency"


Local is 50.9x faster than API for this example.


---

## Part 5: Fine-Tuning on Genomic Datasets

Now we'll put everything together: load a real genomic dataset, tokenize sequences, and fine-tune a classifier. This is the core workflow for adapting foundation models to your research tasks.

### What We'll Do

1. **Load dataset**: Promoter detection task from Hugging Face Hub
2. **Explore data**: Check class balance and sequence statistics
3. **Tokenize**: Convert DNA strings to model-ready tensors
4. **Build classifier**: Add a classification head on top of NT encoder
5. **Train**: Use Hugging Face `Trainer` for easy training
6. **Evaluate**: Check accuracy and confusion matrix

### The Task: Promoter Detection

**Promoters** are DNA regions that initiate gene transcription. Given a 300bp DNA sequence, can the model predict if it contains a promoter?

- **Input**: 300 nucleotide DNA sequence
- **Output**: Binary classification (0 = no promoter, 1 = promoter)

> **Why this matters**: Promoter identification is fundamental for understanding gene regulation and designing synthetic biology constructs.

### 5.1 Hugging Face Authentication

First, authenticate with Hugging Face to access datasets and push models. You'll need a free account and API token.

> **Setup**: Set your token as the `HF_TOKEN` environment variable before running this notebook.

In [15]:
# Login to Hugging Face using your token
# The token should be set as HF_TOKEN environment variable
!huggingface-cli login --token "$HF_TOKEN"

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `test` has been saved to /home/ullahi/.cache/huggingface/stored_tokens
Your token has been saved to /home/ullahi/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [16]:
# Verify login and check library versions
from huggingface_hub import HfApi
import transformers, datasets, sys

print("Library versions:")
print(f"  Transformers: {transformers.__version__}")
print(f"  Datasets:     {datasets.__version__}")
print(f"  Python:       {sys.version.split()[0]}")

# Verify authentication
api = HfApi()
print(f"\nLogged in as: {api.whoami()['name']}")

Library versions:
  Transformers: 4.57.3
  Datasets:     4.5.0
  Python:       3.11.14

Logged in as: ullahi


### 5.2 Load the Nucleotide Transformer Benchmark Dataset

We'll use InstaDeep's benchmark dataset, which contains **18 different genomic classification tasks**:

| Task Type | Examples |
|-----------|----------|
| **Promoter detection** | promoter_all, promoter_tata |
| **Enhancer classification** | enhancers, enhancers_types |
| **Histone marks** | H3, H3K4me1, H3K4me3, ... |
| **Splice sites** | splice_sites_all, acceptors, donors |

For this lab, we'll use the `promoter_all` task.

In [17]:
from datasets import load_dataset

# Load the full dataset from Hugging Face Hub
# Pin a revision for reproducibility (use commit hash in production)
REV = "main"
ds_all = load_dataset(
    "InstaDeepAI/nucleotide_transformer_downstream_tasks",
    revision=REV
)

# View the dataset structure
print("Dataset structure:")
print(ds_all)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['sequence', 'name', 'label', 'task'],
        num_rows: 461850
    })
    test: Dataset({
        features: ['sequence', 'name', 'label', 'task'],
        num_rows: 48797
    })
})


### 5.3 Filter to a Single Task and Create Subset

The dataset contains all 18 tasks together. We'll:
1. Filter to just the `promoter_all` task
2. Take a subset for faster training (15K train, 5K test)

> **Note**: In production, you'd use the full dataset. We use a subset here for faster iteration.

In [18]:
# List all available tasks in the dataset
available_tasks = np.unique(ds_all['train']['task'])
print(f"Available tasks ({len(available_tasks)} total):")
for task in available_tasks:
    print(f"  - {task}")

Available tasks (18 total):
  - H3
  - H3K14ac
  - H3K36me3
  - H3K4me1
  - H3K4me2
  - H3K4me3
  - H3K79me3
  - H3K9ac
  - H4
  - H4ac
  - enhancers
  - enhancers_types
  - promoter_all
  - promoter_no_tata
  - promoter_tata
  - splice_sites_acceptors
  - splice_sites_all
  - splice_sites_donors


In [21]:
from collections import Counter
from datasets import DatasetDict

# Select the task to work with
TASK = "promoter_all"  # Try others: "enhancers", "H3K4me3", "splice_sites_all"

# Filter dataset to selected task
ds_full = ds_all.filter(lambda ex: ex["task"] == TASK)

# Create a smaller subset for faster training
# (Remove these lines to use the full dataset)
ds = DatasetDict({
    "train": ds_full["train"].shuffle(seed=42).select(range(15000)),
    "test":  ds_full["test"].shuffle(seed=42).select(range(5000))
})

# Show dataset sizes
sizes = {k: len(v) for k, v in ds.items()}
print(f"Task: {TASK}")
print(f"Dataset sizes: {sizes}")
print(f"\nFeatures: {ds['train'].features}")

# Check class distribution
y = [ex["label"] for ex in ds["train"]]
print(f"\nClass distribution (train): {Counter(y)}")

Task: promoter_all
Dataset sizes: {'train': 15000, 'test': 5000}

Features: {'sequence': Value('string'), 'name': Value('string'), 'label': Value('int32'), 'task': Value('string')}

Class distribution (train): Counter({0: 7531, 1: 7469})


### 5.4 Explore the Data

Before training, always inspect your data:
- How many samples?
- What's the class balance?
- What are the sequence lengths?

#### How many sequences in training set?

#### Sequence Length Statistics

Understanding sequence lengths helps you choose `max_length` for tokenization:
- Too short → truncates sequences, loses information
- Too long → wastes memory, slows training

In [22]:
import numpy as np

# Calculate sequence lengths for a sample of the data
sample_size = min(5000, len(ds["train"]))
lengths = np.array([len(ex["sequence"]) for ex in ds["train"].select(range(sample_size))])

print("Sequence Length Statistics:")
print(f"  Minimum: {lengths.min()} bp")
print(f"  Maximum: {lengths.max()} bp")
print(f"  Mean:    {lengths.mean():.1f} bp")
print(f"  Std:     {lengths.std():.1f} bp")

# Recommendation for max_length
print(f"\n→ Recommendation: Set MAX_LEN = {int(lengths.max() * 1.5)} to safely accommodate all sequences")

Sequence Length Statistics:
  Minimum: 300 bp
  Maximum: 300 bp
  Mean:    300.0 bp
  Std:     0.0 bp

→ Recommendation: Set MAX_LEN = 450 to safely accommodate all sequences


#### Class Balance

Check if classes are balanced. Imbalanced datasets may need special handling (weighted loss, oversampling, etc.).

In [23]:
# Get label distribution
y = [ex["label"] for ex in ds["train"]]
label_counts = Counter(y)

print("Class distribution (training set):")
total = len(y)
for label, count in sorted(label_counts.items()):
    print(f"  Label {label}: {count:,} ({100*count/total:.1f}%)")

# Check if balanced
ratio = min(label_counts.values()) / max(label_counts.values())
if ratio > 0.8:
    print("\n✓ Classes are reasonably balanced")
else:
    print(f"\n⚠ Classes are imbalanced (ratio: {ratio:.2f})")

Class distribution (training set):
  Label 0: 7,531 (50.2%)
  Label 1: 7,469 (49.8%)

✓ Classes are reasonably balanced


In [24]:
from collections import Counter
import pandas as pd
from IPython.display import display, Markdown

# Count labels in train and test
train_counts = Counter([ex["label"] for ex in ds["train"]])
test_counts  = Counter([ex["label"] for ex in ds["test"]])

# Create a summary DataFrame
labels = sorted(set(train_counts.keys()) | set(test_counts.keys()))
df_summary = pd.DataFrame({
    "Label": labels,
    "Train Count": [train_counts.get(lbl, 0) for lbl in labels],
    "Test Count": [test_counts.get(lbl, 0) for lbl in labels],
})
df_summary["Total"] = df_summary["Train Count"] + df_summary["Test Count"]

# Display as table
print("Dataset Split Summary:")
display(df_summary)

Dataset Split Summary:


,Label,Train Count,Test Count,Total
0,0,7531,2507,10038
1,1,7469,2493,9962


### 5.5 Tokenize the Dataset

Now we convert DNA strings to token IDs that the model can process:

1. Load the tokenizer for our chosen model
2. Apply tokenization to all sequences with padding/truncation
3. Create a validation split from training data

In [25]:
from transformers import AutoTokenizer
from datasets import DatasetDict

# Choose model for tokenization
# We use a smaller model (50M) for faster training
MODEL_ID = "InstaDeepAI/nucleotide-transformer-v2-50m-3mer-multi-species"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
print(f"Tokenizer vocabulary size: {tokenizer.vocab_size}")

# Set max sequence length (in tokens, not nucleotides!)
# For k-mer tokenizers, each token represents ~6 nucleotides
MAX_LEN = 512

def preprocess(batch):
    """Tokenize DNA sequences and prepare for training."""
    # Tokenize sequences
    toks = tokenizer(
        batch["sequence"],
        padding="max_length",      # Pad to MAX_LEN
        truncation=True,           # Truncate if longer than MAX_LEN
        max_length=MAX_LEN
    )
    # Add labels for classification
    toks["labels"] = batch["label"]
    return toks

# Apply tokenization to all splits
print("Tokenizing datasets...")
tokenized = DatasetDict({
    k: v.map(preprocess, batched=True, remove_columns=v.column_names)
    for k, v in ds.items()
})

# Create validation split from training data
if "validation" not in tokenized:
    split = tokenized["train"].train_test_split(test_size=0.1, seed=42)
    tokenized = DatasetDict({
        "train": split["train"],
        "validation": split["test"],
        "test": tokenized["test"]
    })

print("\nTokenized dataset:")
print(tokenized)

Tokenizer vocabulary size: 75
Tokenizing datasets...


Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]


Tokenized dataset:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 13500
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5000
    })
})


In [26]:
# Inspect tokenizer details
print("Tokenizer configuration:")
print(f"  Model max length: {tokenizer.model_max_length}")
print(f"  Padding side: {tokenizer.padding_side}")
print(f"  Special tokens: {tokenizer.special_tokens_map}")

Tokenizer configuration:
  Model max length: 2048
  Padding side: right
  Special tokens: {'unk_token': '<unk>', 'pad_token': '<pad>', 'cls_token': '<cls>', 'mask_token': '<mask>'}


In [27]:
# Example: View tokenized data
sample = tokenized['train'][0]
print("Sample tokenized sequence:")
print(f"  First 10 token IDs: {sample['input_ids'][:10]}")
print(f"  Labels: {sample['labels']}")

Sample tokenized sequence:
  First 10 token IDs: [3, 30, 17, 13, 67, 64, 35, 67, 61, 32]
  Labels: 0


### 5.6 Build the Classifier Model

We'll build a classifier by:
1. Loading the pre-trained NT encoder (the "foundation")
2. Adding a linear classification head on top
3. Wrapping it in a class compatible with HF Trainer

**Architecture**:
```
Input IDs → NT Encoder → [CLS] Token Embedding → Linear Layer → Class Logits
                         (hidden_size)            (num_labels)
```

> **Key Point**: We're using `AutoModel` (encoder only) + custom head, not `AutoModelForSequenceClassification`. This gives us more control.

In [28]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

# Use a 100M parameter model for better performance
MODEL_ID = "InstaDeepAI/nucleotide-transformer-v2-100m-multi-species"

# Load the pre-trained encoder
base_model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Get hidden size from config
hidden_size = base_model.config.hidden_size

# Number of classes in our task
num_labels = len(set(ds["train"]["label"]))

class NTClassifier(nn.Module):
    """
    Custom classifier using Nucleotide Transformer as backbone.
    
    Architecture:
        Input → NT Encoder → CLS token → Linear → Output logits
    """
    def __init__(self, base_model, num_labels, hidden_size):
        super().__init__()
        self.base_model = base_model
        self.classifier = nn.Linear(hidden_size, num_labels)
        self.num_labels = num_labels
    
    def forward(self, input_ids, attention_mask=None, labels=None):
        # Get encoder outputs
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # Pool: use CLS token (first token)
        # Shape: (batch_size, hidden_size)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        
        # Classification head
        logits = self.classifier(pooled_output)
        
        # Compute loss if labels provided
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
        
        # Return dict compatible with HF Trainer
        return {"loss": loss, "logits": logits}

# Create the model
model = NTClassifier(base_model, num_labels, hidden_size)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model created successfully!")
print(f"  Total parameters:     {total_params/1e6:.1f}M")
print(f"  Trainable parameters: {trainable_params/1e6:.1f}M")
print(f"  Hidden size:          {hidden_size}")
print(f"  Number of classes:    {num_labels}")

The following layers were not sharded: pooler.dense.bias, encoder.layer.*.LayerNorm.bias, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.intermediate.dense.bias, contact_head.regression.weight, encoder.layer.*.attention.self.key.weight, encoder.layer.*.attention.self.query.weight, encoder.layer.*.attention.LayerNorm.bias, encoder.layer.*.output.dense.weight, encoder.layer.*.attention.LayerNorm.weight, embeddings.word_embeddings.weight, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.attention.self.value.bias, encoder.layer.*.output.dense.bias, pooler.dense.weight, encoder.emb_layer_norm_after.bias, encoder.layer.*.attention.self.value.weight, encoder.layer.*.attention.self.query.bias, contact_head.regression.bias, encoder.layer.*.attention.self.rotary_embeddings.inv_freq, encoder.layer.*.LayerNorm.weight, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.attention.self.key.bias, encoder.emb_layer_norm_after.weight


RuntimeError: Error(s) in loading state_dict for Linear:
	size mismatch for weight: copying a param with shape torch.Size([4096, 512]) from checkpoint, the shape in current model is torch.Size([2048, 512]).

### 5.7 Train the Model

We use Hugging Face's `Trainer` class which handles:
- Data loading and batching
- Forward/backward passes
- Optimization (AdamW)
- Learning rate scheduling
- Checkpointing
- Evaluation

> **Note**: Training will take a few minutes on GPU. Watch the loss decrease!

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

# Load evaluation metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    """Compute accuracy and F1 score for evaluation."""
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

# Training configuration
args = TrainingArguments(
    output_dir="nt_promoter_run",       # Where to save checkpoints
    per_device_train_batch_size=16,      # Batch size (reduce if OOM)
    per_device_eval_batch_size=16,
    learning_rate=2e-5,                  # Learning rate for AdamW
    num_train_epochs=2,                  # Number of epochs
    eval_strategy="epoch",               # Evaluate after each epoch
    save_strategy="no",                  # Don't save checkpoints
    logging_steps=50,                    # Log every 50 steps
    report_to="none",                    # Disable W&B/tensorboard
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Train!
print("Starting training...")
train_out = trainer.train()

# Evaluate on validation set
print("\nValidation results:")
eval_out = trainer.evaluate()
for k, v in eval_out.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

### 5.8 Evaluate on Test Set

Final evaluation on the held-out test set. This is the true measure of model performance.

We'll look at:
- **Accuracy**: Overall correct predictions
- **Precision/Recall/F1**: Per-class performance
- **Confusion Matrix**: Where the model makes mistakes

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Run predictions on test set
print("Evaluating on test set...")
pred = trainer.predict(tokenized["test"])

# Extract predictions and labels
y_true = pred.label_ids
y_pred = pred.predictions.argmax(-1)

# Print classification report
print("\n" + "="*50)
print("Classification Report")
print("="*50)
print(classification_report(y_true, y_pred, digits=4))

# Print confusion matrix
print("Confusion Matrix:")
cm = confusion_matrix(y_true, y_pred)
print(cm)
print("\n  (rows = true labels, cols = predicted labels)")
print(f"  Diagonal = correct predictions: {cm.diagonal().sum()}/{len(y_true)}")

---

## Exercises: Experiment and Explore!

Now that you've completed the core workflow, try these experiments:

### 1. Try a Different Task
Change `TASK = "promoter_all"` to one of:
- `"enhancers"` - Enhancer vs non-enhancer
- `"H3K4me3"` - Histone modification prediction
- `"splice_sites_all"` - Splice site detection

### 2. Adjust Hyperparameters
- **Learning rate**: Try `1e-5`, `5e-5`, or `3e-4`
- **Batch size**: Larger batches (32, 64) may improve stability
- **Epochs**: More epochs (3-5) may improve performance

### 3. Try Different Models
- `InstaDeepAI/nucleotide-transformer-v2-50m-3mer-multi-species` (faster)
- `InstaDeepAI/nucleotide-transformer-500m-human-ref` (more powerful)

### 4. Use PEFT/LoRA (Advanced)
Tomorrow we'll explore parameter-efficient fine-tuning, which:
- Trains only ~1% of parameters
- Reduces memory usage significantly
- Often achieves similar performance

---

## Summary

In this lab, you learned to:

1. **Encode DNA** as PyTorch tensors (integer and one-hot)
2. **Load pre-trained models** from Hugging Face Hub
3. **Use the Inference API** for quick experimentation
4. **Fine-tune a classifier** on real genomic data
5. **Evaluate performance** with standard metrics

**Next**: Day 2 will cover PEFT/LoRA for efficient fine-tuning!